# Swan — Final Catalogue for all 2 concerns 

## Inputs
| File | What |
|---|---|
| `all_routine_candidates.csv` | source products (699; 494 without serums) |
| `Swan_CosIng_Ingredient_Reference_2026-10-26.csv` | INCI reference |
| `ingredient_matrix_inci_normalized_v1.csv` | Lina's matrix (= my `draft_for_lina_ingredient_matrix.csv`) |
| `moisturizer_catalogue_keep_reviewed_v1.csv` | the 152 moisturizers Lina kept |
| `moisturizer_catalogue_review_master_v1.csv` | Lina's decisions on 224 moisturizers |
| `moisturizer_inclusion_policy_v1.csv` | Lina's inclusion rules P01–P07 |
| `moisturizers_review_all.csv`, `products_checked_by_hand.csv` | my manual review (03–04/11) |
| `decisions_lina_vs_loredana.csv` | my final decisions on our 31 disagreements |

## Outputs
`catalogue_v2.csv` (one row per product) and `ingredient_matrix_v2.csv` (one column per ingredient).

In [531]:
import pandas as pd

RAW = "../raw_data/"
NB_ROUTINE_PRODUCT = 3
ROUTINE_STEPS = ["cleanser", "moisturizer", "sunscreen"]
MIN_INGREDIENTS = 5
BLEMISH_REFERRAL_LEVEL = 4
LIGHT_SKIN_TYPES = ["oily", "combination", "normal", "not_sure"]

# Word lists for the signals — names from EU Regulation 1223/2009, Annex III (fragrance allergens)
FRAGRANCE_ALLERGENS = ["linalool", "limonene", "citronellol", "geraniol", "citral", "coumarin", "eugenol",
                       "benzyl alcohol", "benzyl benzoate", "benzyl salicylate", "hexyl cinnamal", "farnesol",
                       "alpha-isomethyl ionone", "hydroxycitronellal", "cinnamal", "amyl cinnamal", "isoeugenol"]
ESSENTIAL_OILS = ["lavandula", "citrus", "mentha", "eucalyptus", "rosmarinus", "melaleuca", "pelargonium",
                  "cymbopogon", "cananga", "santalum", "pogostemon", "juniperus", "cinnamomum", "jasminum"]
ACNE_ACTIVES = ["salicylic acid", "niacinamide", "zinc pca", "zinc gluconate", "azelaic acid"]
# Texture: heavy ingredients = butters, waxes, oils, petrolatum (INCI order = amount above 1 %)
HEAVY_INGREDIENTS = ["petrolatum", "paraffinum liquidum", "paraffin", "mineral oil", "butter", "cera", "wax",
                     "lanolin", "squalane", "seed oil", "kernel oil", "fruit oil"]
RICH_NAME_WORDS = ["riche", "rich", "baume", "balm", "cold cream", "nourri"]
LIGHT_NAME_WORDS = ["gel", "fluide", "fluid", "matifiant", "light", "légère"]
ALIASES = {
    "glycérine": "glycerin",
    "propylène glycol": "propylene glycol",
    "xantham gum": "xanthan gum",
    "purified water": "aqua",
    "acohol denat": "alcohol denat.",
    "water": "aqua",
    "eau": "aqua",
    "aqua (water)": "aqua",
    "aqua/water/eau": "aqua",
    "aqua / water": "aqua",
    "aqua / water / eau": "aqua",
    "glycerine": "glycerin",
    "fragrance": "parfum",
    "perfume": "parfum",
    "aroma": "parfum",
    "parfum (fragrance)": "parfum",
    "parfum/fragrance": "parfum",
    "shea butter": "butyrospermum parkii butter",
    "butyrospermum parkii": "butyrospermum parkii butter",
    "alcohol denat": "alcohol denat.",
}

# Load the data 

In [532]:
products = pd.read_csv(RAW + "all_routine_candidates.csv", dtype={"code": str})
cosing = pd.read_csv(RAW + "Swan_CosIng_Ingredient_Reference_2026-10-26.csv", dtype=str)
lina_matrix = pd.read_csv(RAW + "ingredient_matrix_inci_normalized_v1.csv", dtype={"code": str})
lina_keep = pd.read_csv(RAW + "moisturizer_catalogue_keep_reviewed_v1.csv", dtype={"code": str})
lina_master = pd.read_csv(RAW + "moisturizer_catalogue_review_master_v1.csv", dtype={"code": str})
lina_policy = pd.read_csv(RAW + "moisturizer_inclusion_policy_v1.csv")
my_review = pd.read_csv(RAW + "moisturizers_review_all.csv", dtype={"code": str})
my_hand = pd.read_csv(RAW + "products_checked_by_hand.csv", dtype={"code": str})
final_decisions = pd.read_csv(RAW + "decisions_lina_vs_loredana.csv", dtype={"code": str})

files = {"products": products, "cosing": cosing, "lina_matrix": lina_matrix, "lina_keep": lina_keep,
         "lina_master": lina_master, "lina_policy": lina_policy, "my_review": my_review,
         "my_hand": my_hand, "final_decisions": final_decisions}
for name, table in files.items():
    print(name.ljust(16), table.shape)

products         (699, 7)
cosing           (1521, 15)
lina_matrix      (235, 253)
lina_keep        (152, 49)
lina_master      (224, 49)
lina_policy      (7, 6)
my_review        (141, 10)
my_hand          (28, 10)
final_decisions  (244, 16)


In [533]:
# Check role based on data audit provided at the beginning
print(products["role"].value_counts())
print()

full = pd.read_csv(RAW + "en.openbeautyfacts.org.products.csv", sep="\t", low_memory=False,
                   usecols=["code", "product_name", "brands", "categories_tags", "countries_tags", "ingredients_text"],
                   dtype={"code": str})
obf = full[full["product_name"].notna() & full["ingredients_text"].notna()].copy()
print("All products:", len(full))
print("With name + ingredients:", len(obf))

categories = obf["categories_tags"].fillna("")
print("Cleansers:  ", categories.str.contains("cleans|nettoyant|demaquill").sum())
print("Moisturizers:", categories.str.contains("moistur|face-cream|creme").sum())
print("Sunscreens: ", categories.str.contains("sunscreen|sun-protection|solaire").sum())
print("France:     ", obf["countries_tags"].fillna("").str.contains("france").sum())

role
moisturizer        244
treatment_serum    205
cleanser           189
sunscreen           61
Name: count, dtype: int64

All products: 64237
With name + ingredients: 16287
Cleansers:   206
Moisturizers: 420
Sunscreens:  345
France:      7031


# CHECK IS ANY LINK POSSIBLE WITH SEPHORA DATA

In [534]:
import re
import difflib

BRAND_ALIASES = {"kiehls since 1851": "kiehls", "estée lauder": "estee lauder", "lancôme": "lancome"}

def clean(text):
    """Lowercase, no punctuation, single spaces: 'L'Oréal Paris' -> 'loréal paris'."""
    text = str(text).lower().split(",")[0]          # OBF brands: 'Garnier, Garnier Skin Naturals' -> first one
    text = re.sub(r"[^a-z0-9àâçéèêëîïôûùüÿ ]", "", text)
    return " ".join(text.split())

def clean_brand(text):
    key = clean(text)
    return BRAND_ALIASES.get(key, key)

In [535]:
#add sephora data set
sephora = pd.read_csv(RAW + "sephora_product_info.csv")
sephora = sephora[sephora["primary_category"] == "Skincare"].copy()
name = sephora["product_name"].str.lower().fillna("")

truth = {"cleanser": sephora["tertiary_category"].isin(["Face Wash & Cleansers", "Makeup Removers"]),
         "moisturizer": sephora["tertiary_category"].isin(["Moisturizers", "Night Creams"]),
         "sunscreen": sephora["tertiary_category"] == "Face Sunscreen"}
rules = {"cleanser": name.str.contains("nettoyant|cleanser|gel moussant|micellaire"),
         "moisturizer": name.str.contains("crème visage|creme visage|soin de jour|soin de nuit|moisturi"),
         "sunscreen": name.str.contains("spf|fps|solaire")}

for step in rules:
    found = rules[step]
    right = found & truth[step]
    print(step.ljust(12), "found:", found.sum(), "| right step:", right.sum(),
          "| real ones found:", right.sum(), "/", truth[step].sum())

cleanser     found: 164 | right step: 140 | real ones found: 140 / 227
moisturizer  found: 267 | right step: 219 | real ones found: 219 / 403
sunscreen    found: 174 | right step: 94 | real ones found: 94 / 95


In [536]:
obf["brand_key"] = obf["brands"].apply(clean_brand)
obf["name_key"] = obf["product_name"].apply(clean)
sephora["brand_key"] = sephora["brand_name"].apply(clean_brand)
sephora["name_key"] = sephora["product_name"].apply(clean)

common_brands = set(obf["brand_key"]) & set(sephora["brand_key"])
print("Brands in both sources:", len(common_brands))

Brands in both sources: 45


In [537]:
matches = []
for _, product in sephora[sephora["brand_key"].isin(common_brands)].iterrows():
    same_brand = obf[obf["brand_key"] == product["brand_key"]]
    best = difflib.get_close_matches(product["name_key"], list(same_brand["name_key"]), n=1, cutoff=0.75)
    if best:
        obf_row = same_brand[same_brand["name_key"] == best[0]].iloc[0]
        matches.append({"brand": product["brand_name"], "sephora_name": product["product_name"],
                        "obf_name": obf_row["product_name"], "obf_code": obf_row["code"],
                        "sephora_category": product["tertiary_category"],
                        "obf_categories": obf_row["categories_tags"]})

matches = pd.DataFrame(matches)
print("Products found in both:", len(matches))
matches.head(20)

Products found in both: 79


,brand,sephora_name,obf_name,obf_code,sephora_category,obf_categories
0,BeautyBio,The Balance pH Balancing Gel Cleanser,The Balance pH Balancing Gel Cleanser,0689551340072,Face Wash & Cleansers,NaN
1,Clarins,Hydrating Gentle Foaming Cleanser with Aloe Vera,Hydrating Gentle Foaming Cleanser,3380810427325,Face Wash & Cleansers,NaN
2,Clarins,Purifying Gentle Foaming Cleanser with Salicyl...,Purifying Gentle Foaming Cleanser,3380810427318,Face Wash & Cleansers,NaN
3,CLINIQUE,Dramatically Different Moisturizing Gel,Clinique Dramatically Different Moisturizing L...,0020714598907,Moisturizers,"nl:beauty,nl:dagcreme,nl:skincare"
4,CLINIQUE,All About Clean Liquid Facial Soap,All About Clean Liquid Facial Soap Mild,0695652145638,Face Wash & Cleansers,NaN
5,CLINIQUE,Dramatically Different Moisturizing Lotion+,Clinique Dramatically Different Moisturizing L...,0020714598907,Moisturizers,"nl:beauty,nl:dagcreme,nl:skincare"
6,CLINIQUE,Smart Clinical Repair Wrinkle Correcting Eye C...,Clinique Smart Clinical Repair Wrinkle Correct...,0192333102749,Eye Creams & Treatments,"nl:beauty,nl:oogcreme,nl:skincare"
7,CLINIQUE,Smart Clinical Repair Wrinkle Correcting Cream,Clinique Smart Clinical Repair Wrinkle Correct...,0192333102749,Moisturizers,"nl:beauty,nl:oogcreme,nl:skincare"
8,CLINIQUE,Moisture Surge 100H Auto-Replenishing Hydrator...,Clinique Moisture Surge 100H Auto-Replenishing...,0192333066942,Moisturizers,"nl:beauty,nl:dagcreme,nl:skincare"
9,CLINIQUE,Smart Clinical Repair Wrinkle Correcting Serum,Clinique Smart Clinical Repair Wrinkle Correct...,0192333102749,Face Serums,"nl:beauty,nl:oogcreme,nl:skincare"


In [538]:
# the most frequent Open Beauty Facts categories among products with ingredients
obf_categories = obf["categories_tags"].fillna("").str.split(",").explode().str.strip()
obf_categories = obf_categories[obf_categories != ""]
print(obf_categories.value_counts().head(40))

# Sephora skincare categories
print(sephora["tertiary_category"].value_counts())

categories_tags
en:hygiene                    2700
en:open-beauty-facts          2468
en:non-food-products          2249
en:hair                       1472
en:shampoos                   1082
en:showers-and-baths           744
en:face                        698
en:body                        671
en:deodorants                  668
en:soaps                       647
en:shower-gels                 600
en:toothpastes                 510
en:cosmetic-products           499
en:makeup                      323
en:suncare                     319
en:in-sun-protections          292
en:facial-creams               279
en:sunscreen                   268
en:body-creams                 228
en:hand-creams                 204
en:toothpaste                  201
en:cleansers                   172
nl:beauty                      161
en:2-in-1-shampoos             151
en:accessories                 148
en:hair-care                   147
en:hair-conditioners           145
en:perfumes                    132
en:a

In [539]:
def obf_step(categories):
    """Swan step from the Open Beauty Facts categories."""
    text = str(categories)
    if "sunscreen" in text or "sun-protection" in text or "solaire" in text:
        return "sunscreen"
    if "cleans" in text or "nettoyant" in text or "demaquill" in text:
        return "cleanser"
    if "moistur" in text or "face-cream" in text or "creme" in text:
        return "moisturizer"
    return "other"

SEPHORA_STEP = {"Face Wash & Cleansers": "cleanser", "Makeup Removers": "cleanser",
                "Moisturizers": "moisturizer", "Night Creams": "moisturizer",
                "Face Sunscreen": "sunscreen"}

matches["obf_step"] = matches["obf_categories"].apply(obf_step)
matches["sephora_step"] = matches["sephora_category"].map(SEPHORA_STEP).fillna("other")
matches["same_step"] = matches["obf_step"] == matches["sephora_step"]

In [540]:
print("Same step:", matches["same_step"].sum(), "/", len(matches))
print()
print(pd.crosstab(matches["sephora_step"], matches["obf_step"]))

Same step: 38 / 79

obf_step      moisturizer  other
sephora_step                    
cleanser                0     29
moisturizer             5      8
other                   3     33
sunscreen               0      1


In [541]:
cleansers = matches[matches["sephora_step"] == "cleanser"]
with pd.option_context("display.max_colwidth", 120):
    display(cleansers[["sephora_name", "obf_name", "obf_categories"]].head(15))
print("Without any OBF category:", cleansers["obf_categories"].isna().sum(), "/", len(cleansers))

,sephora_name,obf_name,obf_categories
0,The Balance pH Balancing Gel Cleanser,The Balance pH Balancing Gel Cleanser,NaN
1,Hydrating Gentle Foaming Cleanser with Aloe Vera,Hydrating Gentle Foaming Cleanser,NaN
2,Purifying Gentle Foaming Cleanser with Salicylic Acid,Purifying Gentle Foaming Cleanser,NaN
4,All About Clean Liquid Facial Soap,All About Clean Liquid Facial Soap Mild,NaN
10,Rinse-Off Foaming Cleanser,All About Clean Rinse-Off Foaming Cleanser,NaN
13,All About Clean Foaming Facial Soap,All About Clean Liquid Facial Soap Mild,NaN
16,Special Cleansing Gel,Special Cleansing Gel,NaN
21,Advanced Night Micro Cleansing Balm,Advanced Night Micro Cleansing Balm,NaN
22,Advanced Night Micro Cleansing Foam,Advanced Night Micro Cleansing Foam,NaN
23,Deep Cleanser with Red Clay,Pure Skin Deep Cleanser with Red Clay,NaN


Without any OBF category: 28 / 29


In [542]:
with pd.option_context("display.max_colwidth", 60):
    display(matches[~matches["same_step"]][["brand", "sephora_name", "obf_name", "sephora_step", "obf_step"]].head(20))

,brand,sephora_name,obf_name,sephora_step,obf_step
0,BeautyBio,The Balance pH Balancing Gel Cleanser,The Balance pH Balancing Gel Cleanser,cleanser,other
1,Clarins,Hydrating Gentle Foaming Cleanser with Aloe Vera,Hydrating Gentle Foaming Cleanser,cleanser,other
2,Clarins,Purifying Gentle Foaming Cleanser with Salicylic Acid,Purifying Gentle Foaming Cleanser,cleanser,other
4,CLINIQUE,All About Clean Liquid Facial Soap,All About Clean Liquid Facial Soap Mild,cleanser,other
6,CLINIQUE,Smart Clinical Repair Wrinkle Correcting Eye Cream,Clinique Smart Clinical Repair Wrinkle Correcting Eye Cr...,other,moisturizer
9,CLINIQUE,Smart Clinical Repair Wrinkle Correcting Serum,Clinique Smart Clinical Repair Wrinkle Correcting Eye Cr...,other,moisturizer
10,CLINIQUE,Rinse-Off Foaming Cleanser,All About Clean Rinse-Off Foaming Cleanser,cleanser,other
13,CLINIQUE,All About Clean Foaming Facial Soap,All About Clean Liquid Facial Soap Mild,cleanser,other
14,CLINIQUE,Mini Moisture Surge 100H Auto-Replenishing Hydrator Mois...,Clinique Moisture Surge 100H Auto-Replenishing Hydrator ...,other,moisturizer
16,Dermalogica,Special Cleansing Gel,Special Cleansing Gel,cleanser,other


## Exploration conclusion (10/11)

- 79 products found in both Sephora and Open Beauty Facts (45 common brands).
- For the 43 that Sephora puts in a routine step, OBF categories give the right step for 5 (12%): most OBF records have no category at all (all 15 cleansers checked: empty)

In [543]:
# a code must be unique in each file, except in products (see below)
for name in ["lina_matrix", "lina_keep", "lina_master", "my_review", "my_hand", "final_decisions"]:
    assert files[name]["code"].is_unique, name
print("Codes unique in every decision file")

# products: some codes are in two steps (e.g. moisturizer + serum)
twice = products[products["code"].duplicated(keep=False)].sort_values("code")
twice[["code", "role", "product_name"]]

Codes unique in every decision file


,code,role,product_name
460,00611903,sunscreen,Dr.Melaxin Hydro Ceramic Serum Infused Essence...
613,00611903,treatment_serum,Dr.Melaxin Hydro Ceramic Serum Infused Essence...
401,3600541144019,moisturizer,Skin Naturals Ultra Lift + Sérum Crème Soin an...
540,3600541144019,treatment_serum,Skin Naturals Ultra Lift + Sérum Crème Soin an...
417,4005900107589,moisturizer,Perles de Q10 sérum concentré
511,4005900107589,treatment_serum,Perles de Q10 sérum concentré
299,7640183490712,moisturizer,Ultra Sensitive
597,7640183490712,treatment_serum,Ultra Sensitive
298,7640183491009,moisturizer,Cellular Night Repair Miracle Without Essentia...
588,7640183491009,treatment_serum,Cellular Night Repair Miracle Without Essentia...


# CHECK & VALIDATE THE CLEANSERS

### Check the amount of cleansers in FR and EU 

In [544]:
cleansers = products[products["role"] == "cleanser"].copy()
print("Cleansers:", len(cleansers))
print("In France:", cleansers["france"].sum())
print("In the EU:", cleansers["eu"].sum())

Cleansers: 189
In France: 89
In the EU: 114


In [545]:
# one list per rule: rule_id, product_category, decision, definition, source, approval_status
rules = [
    ["C01", "Face cleansers", "Include",
     "Gels, foams, milks, oils, balms and creams whose main use is to cleanse or remove make-up from the face.",
     "Audit 27/09", "Proposed by Loredana, to validate"],
    ["C02", "Micellar waters for the face", "Include",
     "Micellar or cleansing waters for the face, or for face and eyes.",
     "Audit 27/09", "Proposed by Loredana, to validate"],
    ["C03", "Eye-only make-up removers", "Exclude",
     "Products only for the eyes or lashes: they do not cleanse the face.",
     "Audit 27/09", "Proposed by Loredana, to validate"],
    ["C04", "Wipes and pads", "Exclude",
     "Single-use or washable wipes, pads and discs: an accessory or a format we can't compare with the other cleansers.",
     "Audit 27/09 + Loredana", "Proposed by Loredana, to validate"],
    ["C05", "Scrubs and peelings", "Exclude",
     "Products whose main use is to exfoliate (scrub particles, peeling, enzyme powder): a treatment, not a gentle daily cleanser. Exfoliating cleansers with an acid (e.g. salicylic acid cleanser) are not C05.",
     "Loredana", "Decided by Loredana 10/11"],
    ["C06", "Masks, clay, nose strips, spot patches", "Exclude",
     "Not a daily cleanser.",
     "Audit 27/09 + Loredana", "Proposed by Loredana, to validate"],
    ["C07", "Solid and liquid soaps", "Include only if made for the face",
     "Soap bars and liquid soaps: kept only when the product says it is for the face, hand and body soaps are excluded. Checked by hand.",
     "Loredana", "Decided by Loredana 10/11"],
    ["C08", "Not a skin cleanser", "Exclude",
     "Nail polish removers (name, or acetone first / diethyl succinate / ethyl acetate in the ingredients), household products, intimate washes.",
     "Audit 27/09 + Loredana", "Proposed by Loredana, to validate"],
    ["C09", "Toners", "Exclude",
     "Toning lotions are a separate step, not a cleanser.",
     "Audit 27/09", "Proposed by Loredana, to validate"],
    ["C10", "Leave-on serums and lotions", "Exclude",
     "Serums and lotions filed as cleansers but applied after cleansing: another step of the routine.",
     "Hand check 10/11", "Proposed by Loredana, to validate"],
]

policy = pd.DataFrame(rules, columns=["rule_id", "product_category", "decision", "definition",
                                      "source", "approval_status"])
policy["policy_version"] = "v1"
policy.to_csv(RAW + "cleanser_inclusion_policy_v1.csv", index=False)
policy[["rule_id", "product_category", "decision"]]

,rule_id,product_category,decision
0,C01,Face cleansers,Include
1,C02,Micellar waters for the face,Include
2,C03,Eye-only make-up removers,Exclude
3,C04,Wipes and pads,Exclude
4,C05,Scrubs and peelings,Exclude
5,C06,"Masks, clay, nose strips, spot patches",Exclude
6,C07,Solid and liquid soaps,Include only if made for the face
7,C08,Not a skin cleanser,Exclude
8,C09,Toners,Exclude
9,C10,Leave-on serums and lotions,Exclude


## Set the rule for each product

In [546]:
# normalize the cleansers in europe and generate a policy file
CLEANSER_WORDS = ["clean", "nettoy", "wash", "lavant", "démaquill", "demaquill", "micel", "h2o", "h20",
                  "foam", "mousse", "purif", "temizle", "yıkama", "reinig", "limpia", "limpeza",
                  "desmaquill", "puhdistus", "умыв"]
EYE_WORDS = ["eye", "eyes", "yeux", "augen", "cils"]
FACE_WORDS = ["face", "visage", "gesicht"]

# C08 not a skin cleanser
NOT_COSMETIC = ["dissolvant", "nail", "nagel", "ongles", "quitaesmalte", "acetona",
                "loiça", "wood", "desinfektion", "intimo", "intime"]
NAIL_INGREDIENTS = ["diethyl succinate", "ethyl acetate", "butyl acetate"]
# C05 scrubs, C06 not a daily cleanser, C04 wipes, C07 soaps
SCRUB = ["scrub", "peeling", "enzyme powder"]
NOT_DAILY = ["strips", "bandes nez", "t-bant", "patch", "argile"]
WIPES = ["wipe", "lingette", "pads"]
SOAP = ["soap", "savon", "sabun"]

In [547]:
def has_word(text, words):
    """True if one of the words is in the text."""
    for word in words:
        if word in text:
            return True
    return False


def policy_rule(row):
    """The rule of the policy for one product. The first rule that matches wins."""
    name = str(row["product_name"]).lower()
    inci = str(row["ingredients_text"]).lower()
    first_ingredient = inci.split(",")[0].strip()

    if has_word(name, NOT_COSMETIC) or first_ingredient == "acetone" or has_word(inci, NAIL_INGREDIENTS):
        return "C08"
    if " ou " in name:
        return "to check by hand"      # two products in one record
    if has_word(name, SCRUB):
        return "C05"
    if has_word(name, NOT_DAILY):
        return "C06"
    if ("tonique" in name or "tonic" in name) and "lait" not in name:
        return "C09"
    if has_word(name, EYE_WORDS) and not has_word(name, FACE_WORDS):
        return "C03"
    if has_word(name, WIPES):
        return "C04"
    if has_word(name, SOAP) or "bar" in name.split():
        return "C07"
    if has_word(name, CLEANSER_WORDS):
        return "C01"
    return "to check by hand"


cleansers["policy_rule"] = cleansers.apply(policy_rule, axis=1)
pd.crosstab(cleansers["policy_rule"], cleansers["france"], margins=True)

france,False,True,All
policy_rule,,,
C01,59,46,105
C03,8,10,18
C04,2,2,4
C05,3,0,3
C06,3,3,6
C07,12,0,12
C08,9,18,27
C09,0,1,1
to check by hand,4,9,13


In [548]:
# decision of each rule, read from the policy
DECISION = {}
for index, row in policy.iterrows():
    if row["decision"] == "Include only if made for the face":
        DECISION[row["rule_id"]] = "To check"          # C07: checked by hand
    else:
        DECISION[row["rule_id"]] = row["decision"]
DECISION["to check by hand"] = "To check"
print(DECISION)

cleansers["policy_decision"] = cleansers["policy_rule"].map(DECISION)
assert cleansers["policy_decision"].isna().sum() == 0, "a rule has no decision"
print(cleansers["policy_decision"].value_counts())

{'C01': 'Include', 'C02': 'Include', 'C03': 'Exclude', 'C04': 'Exclude', 'C05': 'Exclude', 'C06': 'Exclude', 'C07': 'To check', 'C08': 'Exclude', 'C09': 'Exclude', 'C10': 'Exclude', 'to check by hand': 'To check'}
policy_decision
Include     105
Exclude      59
To check     25
Name: count, dtype: int64


In [549]:
# look at the automatic exclusions
for rule in ["C03", "C04", "C05", "C06", "C08", "C09"]:
    print("=====", rule)
    group = cleansers[cleansers["policy_rule"] == rule]
    for index, row in group.iterrows():
        print(row["france"], "|", row["product_name"], "|", row["brands"])

===== C03
False | Eye Make-up Remover | Nivea
False | Simple Eye Make Up Remover Pads | Bonaventura sales
True | Cils démasq | Maybelline New York
False | Balea Augen Make-up Entferner ölfrei, 100 ml | Balea
False | Mizellen Augen Make-Up Entferner Pads | Balea
True | Démaquillant yeux peaux sensibles | La Roche-Posay
False | Sanfter Augen Make-up Entferner | Cadea Vera
False | Augen Make-Up Entferner | Alverde Naturkosmetik
False | Oil-free eye makeup remover | Nutrogena
False | Oil-Free Eye-Makeup Remover Pads | Marcelle
True | Le démaquillant express yeux | Yves Rocher
True | Cils Demasq - Lotion fraîcheur démaquillant douceur pour les yeux | Maybelline, Gemey, Gemey Maybelline
True | Bye bye eye makeup | HEMA
True | Eau démaquillante Yeux | Netto
True | Gelée démaquillante yeux | Inell
True | Démaquillant yeux waterproof à l'extrait de prune | By U, Système U
True | Démaquillant Yeux Douceur | Leader Price
True | Démaquillant yeux | Diadermine
===== C04
False | Maximum Strength Cle

## Recheck Data Manually before final validation

In [550]:
to_check = cleansers[cleansers["policy_decision"] == "To check"].copy()
to_check = to_check[["code", "product_name", "brands", "france", "eu", "policy_rule", "ingredients_text"]]
print(len(to_check), "products to check")

# the Open Beauty Facts page of each product (photos of the pack)
to_check["link"] = "https://world.openbeautyfacts.org/product/" + to_check["code"]
for index, row in to_check.iterrows():
    print(row["product_name"], "→", row["link"])

25 products to check
CeraVe → https://world.openbeautyfacts.org/product/3606000537750
Zapuszkowany → https://world.openbeautyfacts.org/product/5901501089059
Dr. Ea Salis Gözenek Sıkılaştırıcı Sabun 70 gr → https://world.openbeautyfacts.org/product/8697853217857
Mizellen → https://world.openbeautyfacts.org/product/3600541744554
Effaclar → https://world.openbeautyfacts.org/product/3337875722827
Dove Nemlendirici Sıvı Sabun Caring → https://world.openbeautyfacts.org/product/8683130052129
Respectissime → https://world.openbeautyfacts.org/product/3433422401907
hautrein → https://world.openbeautyfacts.org/product/4066447492804
Moisturizing Aloe Vera & Jasmine Cream Soap - Box → https://world.openbeautyfacts.org/product/7640183490026
Allergic Skin Cleansing Bar - Bag → https://world.openbeautyfacts.org/product/7640183490286
Make-Up Remover Cleansing Bar - Solid Box → https://world.openbeautyfacts.org/product/7640183490262
Charcoal & Dead Sea Salt Detox Soap - Solid Box → https://world.openbea

In [551]:
# Final decision after manual hand check
MY_DECISIONS = [
    ["CeraVe", "Include", "C01", "INCI = CeraVe Foaming Facial Cleanser, rinse-off face gel (source: CeraVe page with INCI)"],
    ["Zapuszkowany", "Include", "C01", "face make-up removing wax (source: product page + label photo)"],
    ["Dr. Ea Salis Gözenek Sıkılaştırıcı Sabun 70 gr", "Include", "C07", "soap bar for the face (pore tightening, mandelic acid)"],
    ["Mizellen", "Include", "C02", "bi-phase micellar water for face, eyes and lips, formula code Z289832/1 (source: page with formula code)"],
    ["Effaclar", "Exclude", "C10", "INCI = Effaclar Ultra Concentrated Serum, a leave-on peeling care (source: official page)"],
    ["Dove Nemlendirici Sıvı Sabun Caring", "Exclude", "C07", "liquid hand soap (source: Dove Caring page)"],
    ["Respectissime", "Exclude", "C03", "waterproof eye make-up remover (source: official page)"],
    ["hautrein", "Include", "C01", "Anti Pickel Waschgel Hautrein, rinse-off face gel (source: dm, owner of Balea)"],
    ["Moisturizing Aloe Vera & Jasmine Cream Soap - Box", "Exclude", "C07", "soap bar, face not stated"],
    ["Allergic Skin Cleansing Bar - Bag", "Exclude", "C07", "soap bar, face not stated"],
    ["Make-Up Remover Cleansing Bar - Solid Box", "Include", "C07", "soap bar sold as a face make-up remover"],
    ["Charcoal & Dead Sea Salt Detox Soap - Solid Box", "Exclude", "C07", "soap bar, face not stated"],
    ["Moisturising Rose & Lavender Cream Soap - Box", "Exclude", "C07", "soap bar, face not stated"],
    ["Charcoal & Dead Sea Salt Detox Soap - Bag", "Exclude", "C07", "soap bar, face not stated"],
    ["Moisturizing Aloe Vera & Jasmine Cream Soap - Bag", "Exclude", "C07", "soap bar, face not stated"],
    ["Moisturising Rose & Lavender Cream Soap - Bag", "Exclude", "C07", "soap bar, face not stated"],
    ["Allergic Skin Cleansing Bar - Solid Box", "Exclude", "C07", "soap bar, face not stated"],
    ["Make-Up Remover Cleansing Bar - Bag", "Exclude", "duplicate", "same product as the Solid Box record"],
    ["Lait démaquillant Douceur OU Lotion tonique Douceur", "Verify", "C01", "two products in one record: the list (ethylhexyl stearate) looks like the milk"],
    ["Baume Divin", "Include", "C01", "face make-up removing balm (source: archived product page)"],
    ["Demakup Cotton Science Fraicheur", "Exclude", "C04", "soaked cleansing wipes (source: official page with INCI)"],
    ["Eau extraordinaire Rose", "Exclude", "C10", "lotion-serum applied after cleansing (source: archived page + Melvita directions)"],
    ["N°110 Gelée de beauté", "Include", "C01", "leave-on make-up removing gel for face and eyes (source: product page of the time)"],
    ["Trixéra+", "Include", "C01", "barcode = TriXéra+ Sélectiose emollient cleansing gel, rinse-off (source: page with barcode)"],
    ["Lait de toilette très doux Vitamine E Huile d'amande douce", "Include", "C01", "no-rinse cleansing milk, adult use stated on another barcode with the same formula B35849/1"],
]

review = pd.DataFrame(MY_DECISIONS, columns=["product_name", "my_decision", "my_rule", "my_reason"])

# checks: each name is found exactly once, and every product to check has a decision
for name in review["product_name"]:
    found = (to_check["product_name"] == name).sum()
    assert found == 1, name + " found " + str(found) + " times"
assert len(review) == len(to_check), "a product to check has no decision"

# add the code (from the notebook, so it stays exact) and save
review = to_check[["code", "product_name"]].merge(review, on="product_name")
review.to_csv(RAW + "cleanser_review_v1.csv", index=False)
print(review["my_decision"].value_counts())

my_decision
Exclude    14
Include    10
Verify      1
Name: count, dtype: int64


In [552]:
review = pd.read_csv(RAW + "cleanser_review_v1.csv", dtype={"code": str})
review = review[["code", "my_decision", "my_rule", "my_reason"]]

cleansers = cleansers.merge(review, on="code", how="left")


def final_decision(row):
    """My decision when I checked by hand, otherwise the decision of the policy."""
    if row["policy_decision"] == "To check":
        return row["my_decision"]
    return row["policy_decision"]


cleansers["final_decision"] = cleansers.apply(final_decision, axis=1)
pd.crosstab(cleansers["final_decision"], cleansers["france"], margins=True)

france,False,True,All
final_decision,,,
Exclude,35,38,73
Include,65,50,115
Verify,0,1,1
All,100,89,189


In [553]:
# checks
assert len(cleansers) == 189, "a product was lost or added by the merge"
assert cleansers["code"].duplicated().sum() == 0, "a code appears twice"
allowed = ["Include", "Exclude", "Verify"]
wrong = cleansers[~cleansers["final_decision"].isin(allowed)]
assert len(wrong) == 0, wrong["product_name"].tolist()

cleansers.to_csv(RAW + "cleansers_reviewed_v1.csv", index=False)
print("Saved:", len(cleansers), "cleansers")

Saved: 189 cleansers


# Sunscreen Check Quality @ Define Target

## 3. Sunscreens: which ones are real face sunscreens?
61 sunscreens (all countries, 14 in France). Minimum SPF 30: NHS and American Academy of Dermatology.
Version 1: the routine always has a separate sunscreen step. A moisturizer with SPF stays a moisturizer

In [554]:
sunscreens = products[products["role"] == "sunscreen"].copy()
print("Sunscreens:", len(sunscreens), "| France:", sunscreens["france"].sum(), "| EU:", sunscreens["eu"].sum())

# one list per rule: rule_id, product_category, decision, definition, source, approval_status
sun_rules = [
    ["S01", "Dedicated facial sunscreen", "Include",
     "UV protection is the primary advertised use and the product is intended for the face. Tinted and serum formats may qualify.",
     "Draft (research 10/11)", "Proposed by Loredana, to validate"],
    ["S02", "Dedicated face and body sunscreen", "Include",
     "UV protection is the primary advertised use and the directions include the face as well as the body.",
     "Draft (research 10/11)", "Proposed by Loredana, to validate"],
    ["S03", "Other product or application area", "Exclude",
     "Makeup such as BB or CC cream, lip-only or body-only products, after-sun care and self-tanners.",
     "Draft (research 10/11)", "Proposed by Loredana, to validate"],
    ["S04", "Skincare with secondary SPF", "Move to moisturizer",
     "A moisturizer or targeted skincare product with SPF stays in its primary category (Lina's P02).",
     "Draft (research 10/11)", "Proposed by Loredana, to validate"],
    ["S05", "Declared SPF below 30", "Exclude",
     "NHS and American Academy of Dermatology: at least SPF 30.",
     "NHS, AAD", "Proposed by Loredana, to validate"],
    ["S06", "Unresolved sunscreen candidate", "To check",
     "No UV filter in the ingredient list, or the exact variant, primary use, facial use, SPF or UVA protection needs confirmation.",
     "Draft (research 10/11)", "Proposed by Loredana, to validate"],
]

sun_policy = pd.DataFrame(sun_rules, columns=["rule_id", "product_category", "decision", "definition",
                                              "source", "approval_status"])
sun_policy["policy_version"] = "v1"
sun_policy.to_csv(RAW + "sunscreen_inclusion_policy_v1.csv", index=False)
sun_policy[["rule_id", "product_category", "decision"]]

Sunscreens: 61 | France: 14 | EU: 27


,rule_id,product_category,decision
0,S01,Dedicated facial sunscreen,Include
1,S02,Dedicated face and body sunscreen,Include
2,S03,Other product or application area,Exclude
3,S04,Skincare with secondary SPF,Move to moisturizer
4,S05,Declared SPF below 30,Exclude
5,S06,Unresolved sunscreen candidate,To check


In [555]:
# UV filters allowed in the EU (Annex VI of Regulation 1223/2009): INCI names and common names
UV_FILTERS = ["octocrylene", "butyl methoxydibenzoylmethane", "avobenzone", "titanium dioxide", "zinc oxide",
              "ethylhexyl methoxycinnamate", "octyl methoxycinnamate", "homosalate", "ethylhexyl salicylate",
              "octyl salicylate", "bis-ethylhexyloxyphenol", "bemotrizinol", "ethylhexyl triazone",
              "diethylamino hydroxybenzoyl", "methylene bis-benzotriazolyl", "drometrizole trisiloxane",
              "terephthalylidene", "phenylbenzimidazole", "benzophenone-3", "tris-biphenyl triazine",
              "diethylhexyl butamido triazone", "polysilicone-15", "isoamyl p-methoxycinnamate",
              "methoxypropylamino cyclohexenylidene", "methylbenzylidene camphor"]
SUN_WORDS = ["spf", "fps", "sun", "solaire", "solare", "solcreme", "güneş", "uv", "protect"]
# S03 other product or area
OTHER_AREA = ["lip", "lèvre", "levre", "stick", "bb cream", "cc cream", "cc krem",
              "after sun", "après-soleil", "autobronz", "self tan"]
# S05 SPF below 30 written in the name
LOW_SPF = ["spf 6", "spf 10", "spf 15", "spf 20", "spf 25", "spf6", "spf10", "spf15", "spf20", "spf25",
           "fps 10", "fps 15", "fps 20", "fps 25"]
FACE_AND_BODY = ["body", "corps", "famille"]

In [556]:
def sun_policy_rule(row):
    """The rule of the policy for one sunscreen. The first rule that matches wins."""
    name = str(row["product_name"]).lower()
    inci = str(row["ingredients_text"]).lower()

    if has_word(name, OTHER_AREA):
        return "S03"
    if has_word(name, LOW_SPF):
        return "S05"
    if not has_word(inci, UV_FILTERS):
        return "S06"
    if has_word(name, FACE_AND_BODY):
        return "S02"
    if has_word(name, SUN_WORDS):
        return "S01"
    return "to check by hand"

sunscreens["policy_rule"] = sunscreens.apply(sun_policy_rule, axis=1)
pd.crosstab(sunscreens["policy_rule"], sunscreens["france"], margins=True)

france,False,True,All
policy_rule,,,
S01,33,6,39
S02,3,1,4
S03,2,5,7
S05,1,0,1
S06,4,1,5
to check by hand,4,1,5
All,47,14,61


In [557]:
SUN_DECISION = {}
for index, row in sun_policy.iterrows():
    SUN_DECISION[row["rule_id"]] = row["decision"]
SUN_DECISION["to check by hand"] = "To check"
print(SUN_DECISION)

sunscreens["policy_decision"] = sunscreens["policy_rule"].map(SUN_DECISION)
assert sunscreens["policy_decision"].isna().sum() == 0, "a rule has no decision"
print(sunscreens["policy_decision"].value_counts())

{'S01': 'Include', 'S02': 'Include', 'S03': 'Exclude', 'S04': 'Move to moisturizer', 'S05': 'Exclude', 'S06': 'To check', 'to check by hand': 'To check'}
policy_decision
Include     43
To check    10
Exclude      8
Name: count, dtype: int64


In [558]:
# look at the automatic exclusions
for rule in ["S03", "S05"]:
    print("=====", rule)
    group = sunscreens[sunscreens["policy_rule"] == rule]
    for index, row in group.iterrows():
        print(row["france"], "|", row["product_name"], "|", row["brands"])

===== S03
False | Farmasi CC Krem 02 Light to Medium SPF 25 | Farmasi
False | Lip balm classic spf 50+ sunscreen | Ombra
True | BB cream texture légère perfecteur de teint - 01 beige nude | Léa Nature, So' Bio étic
True | BB Cream Mon lait d'Ânesse  SPF 10 - 01 beige lumière | Léa Nature, So' Bio étic
True | BB cream radiance 9 en 1 SPF 25 - 002 medium | Rimmel
True | BB cream radiance 9 en 1 SPF 25 - 00? très claire | Rimmel
True | BB cream radiance 9 en 1 SPF 20 - 001 claire | Rimmel
===== S05
False | Tinted Facial Sunscreen SPF 20 | Beauty By Earth


In [559]:
sun_to_check = sunscreens[sunscreens["policy_decision"] == "To check"].copy()
sun_to_check = sun_to_check[["code", "product_name", "brands", "france", "eu", "policy_rule", "ingredients_text"]]
print(len(sun_to_check), "products to check")

# the Open Beauty Facts page of each product (photos of the pack)
sun_to_check["link"] = "https://world.openbeautyfacts.org/product/" + sun_to_check["code"]
for index, row in sun_to_check.iterrows():
    print(row["france"], "|", row["product_name"], "→", row["link"])

10 products to check
False | Hamilton everyday face → https://world.openbeautyfacts.org/product/9313501072553
False | Nivea Q10 Power Kırışıklık Karşıtı ve Sıkılaştırıcı Gündüz Yüz Bakım Kremi 50 ml → https://world.openbeautyfacts.org/product/4005900546272
False | Moisturizing Sunscreen Lotion For Face and Body 30 → https://world.openbeautyfacts.org/product/0860121000408
True | Crème anti-taches - vitamine c + niacinamide → https://world.openbeautyfacts.org/product/3600551156699
True | Lait Famille - Visage & Corps → https://world.openbeautyfacts.org/product/0673914115674
False | Hada Labo Tokyo Sun Güneş Yüz Kremi 50SPF 50 ml → https://world.openbeautyfacts.org/product/5900525062819
False | Cire Aseptine Yüz Güneş Koruyucu Krem Karma ve Yağlı Ciltler 50+ SPF 50 ml → https://world.openbeautyfacts.org/product/8682276606074
False | Bee Beauty Koruyucu Nemlendirici Yüz Kremi 50 ml → https://world.openbeautyfacts.org/product/2050001490049
False | INGU Tri-peptide Balm Nude Pink → https://w

In [560]:
# my decisions for the sunscreens checked by hand
SUN_MY_DECISIONS = [
    ["Hamilton everyday face", "Include", "S01", "face sunscreen. The export lists only the 4 UV filters: full INCI to get (source: product page)"],
    ["Nivea Q10 Power Kırışıklık Karşıtı ve Sıkılaştırıcı Gündüz Yüz Bakım Kremi 50 ml", "Move to moisturizer", "S04", "anti-wrinkle day cream SPF 30: care first, so a moisturizer (source: page with the exact barcode)"],
    ["Moisturizing Sunscreen Lotion For Face and Body 30", "Include", "S02", "face and body sunscreen SPF 30. US label: UV filters listed apart as Active Ingredients, to add from the label (source: official page)"],
    ["Crème anti-taches - vitamine c + niacinamide", "Move to moisturizer", "S04", "anti-dark spot care SPF 20: care first, so a moisturizer (source: page with the exact barcode)"],
    ["Lait Famille - Visage & Corps", "Include", "S02", "face and body sun milk. No UV filter in the exported list: check if actives are declared apart and confirm the France version (source: Canadian page)"],
    ["Hada Labo Tokyo Sun Güneş Yüz Kremi 50SPF 50 ml", "Include", "S01", "face sunscreen SPF 50. The ingredient field only says Suya Dayanıklı (water resistant): INCI to get (source: page with the exact barcode)"],
    ["Cire Aseptine Yüz Güneş Koruyucu Krem Karma ve Yağlı Ciltler 50+ SPF 50 ml", "Include", "S01", "face sunscreen SPF 50+. Same INCI field problem: INCI to get (source: page with the exact barcode)"],
    ["Bee Beauty Koruyucu Nemlendirici Yüz Kremi 50 ml", "Include", "S01", "SPF 50, applied on face and neck before sun exposure. Same INCI field problem: INCI to get (source: page with the exact barcode)"],
    ["INGU Tri-peptide Balm Nude Pink", "Exclude", "S03", "tinted sun balm for the lips: excluded for the application area (source: INGU lip range, Nude Pink variant)"],
    ["Reginald Helios", "Include", "S01", "brand page confirms sun use, 3 UV filters in the exported INCI. SPF and UVA of the exact code to check on the label (source: Reginald page)"],
]

# products whose ingredient list must be completed before the ingredient-based ranking
INCI_TO_COMPLETE = ["Hamilton everyday face", "Moisturizing Sunscreen Lotion For Face and Body 30",
                    "Lait Famille - Visage & Corps", "Hada Labo Tokyo Sun Güneş Yüz Kremi 50SPF 50 ml",
                    "Cire Aseptine Yüz Güneş Koruyucu Krem Karma ve Yağlı Ciltler 50+ SPF 50 ml",
                    "Bee Beauty Koruyucu Nemlendirici Yüz Kremi 50 ml"]

sun_review = pd.DataFrame(SUN_MY_DECISIONS, columns=["product_name", "my_decision", "my_rule", "my_reason"])

# checks: each name is found exactly once, and every product to check has a decision
for name in sun_review["product_name"]:
    found = (sun_to_check["product_name"] == name).sum()
    assert found == 1, name + " found " + str(found) + " times"
assert len(sun_review) == len(sun_to_check), "a product to check has no decision"

# add the code (from the notebook, so it stays exact) and save
sun_review = sun_to_check[["code", "product_name"]].merge(sun_review, on="product_name")
sun_review.to_csv(RAW + "sunscreen_review_v1.csv", index=False)
print(sun_review["my_decision"].value_counts())

my_decision
Include                7
Move to moisturizer    2
Exclude                1
Name: count, dtype: int64


In [561]:
sun_review = pd.read_csv(RAW + "sunscreen_review_v1.csv", dtype={"code": str})
sun_review = sun_review[["code", "my_decision", "my_rule", "my_reason"]]

sunscreens = sunscreens.merge(sun_review, on="code", how="left")
sunscreens["final_decision"] = sunscreens.apply(final_decision, axis=1)     # same function as the cleansers

# True when the ingredient list must be completed from the label
sunscreens["inci_to_complete"] = sunscreens["product_name"].isin(INCI_TO_COMPLETE)
print("INCI to complete:", sunscreens["inci_to_complete"].sum())
pd.crosstab(sunscreens["final_decision"], sunscreens["france"], margins=True)

INCI to complete: 6


france,False,True,All
final_decision,,,
Exclude,4,5,9
Include,42,8,50
Move to moisturizer,1,1,2
All,47,14,61


In [562]:
# checks
assert len(sunscreens) == 61, "a product was lost or added by the merge"
assert sunscreens["code"].duplicated().sum() == 0, "a code appears twice"
allowed = ["Include", "Exclude", "Verify", "Move to moisturizer"]
wrong = sunscreens[~sunscreens["final_decision"].isin(allowed)]
assert len(wrong) == 0, wrong["product_name"].tolist()

sunscreens.to_csv(RAW + "sunscreens_reviewed_v1.csv", index=False)
print("Saved:", len(sunscreens), "sunscreens")

# products moved to the moisturizers: reviewed again in part 4 (no review by Lina yet)
moved = sunscreens[sunscreens["final_decision"] == "Move to moisturizer"].copy()
moved["role"] = "moisturizer"
moved = moved[["role", "code", "product_name", "brands", "france", "eu", "ingredients_text", "my_reason"]]
moved.to_csv(RAW + "moved_to_moisturizers_v1.csv", index=False)
print("Moved to the moisturizers:", moved["product_name"].tolist())

# France: sunscreens kept, and how many without parfum (for sensitive skin)
kept_fr = sunscreens[(sunscreens["final_decision"] == "Include") & sunscreens["france"]]
no_parfum = 0
for inci in kept_fr["ingredients_text"]:
    text = str(inci).lower()
    if "parfum" not in text and "fragrance" not in text and "aroma" not in text and "perfume" not in text:
        no_parfum = no_parfum + 1
print("Kept in France:", len(kept_fr), "| without parfum:", no_parfum)

Saved: 61 sunscreens
Moved to the moisturizers: ['Nivea Q10 Power Kırışıklık Karşıtı ve Sıkılaştırıcı Gündüz Yüz Bakım Kremi 50 ml', 'Crème anti-taches - vitamine c + niacinamide']
Kept in France: 8 | without parfum: 2


##  Moisturizers finalize the manual check by hand 


In [563]:
moisturizers = products[products["role"] == "moisturizer"].copy()
both = final_decisions[["code", "lina_decision", "my_decision", "my_reason", "conflict", "suggestion",
                        "final_category", "review_policy_annotation"]].rename(
    columns={"my_decision": "loredana_decision", "my_reason": "loredana_reason",
             "final_category": "lina_category", "review_policy_annotation": "lina_rule"})
moisturizers = moisturizers.merge(both, on="code", how="left")

# sunscreens moved here (S04)
sun_moved = sunscreens[sunscreens["final_decision"] == "Move to moisturizer"].copy()
sun_moved["role"] = "moisturizer"
sun_moved["lina_decision"] = "Not reviewed (moved from sunscreens)"
sun_moved["loredana_reason"] = sun_moved["my_reason"]
moisturizers = pd.concat([moisturizers, sun_moved[["role", "code", "product_name", "brands", "france", "eu",
                                                   "ingredients_text", "lina_decision", "loredana_reason"]]],
                         ignore_index=True)
assert moisturizers["code"].is_unique, "a code appears twice"
print("Moisturizers:", len(moisturizers), "| France:", moisturizers["france"].sum(), "| EU:", moisturizers["eu"].sum())

Moisturizers: 246 | France: 115 | EU: 172


In [564]:
def moisturizer_rule(row):
    """What Lina and I decided together, or 'to check by hand'."""
    lina = str(row["lina_decision"])
    me = str(row["loredana_decision"])
    if pd.notna(row["conflict"]):
        return "Disagreement", "To check"
    if lina == "Keep":
        return "Lina and Loredana agree", "Include"
    if lina == "Exclude" or lina.startswith("Excluded at step 2"):
        return "Lina and Loredana agree", "Exclude"
    if lina == "Needs more evidence" and me == "Exclude":
        return "Lina and Loredana agree", "Exclude"
    if lina == "Needs more evidence":
        return "Needs more evidence", "To check"
    return "Moved from sunscreens", "To check"


results = moisturizers.apply(moisturizer_rule, axis=1)
moisturizers["policy_rule"] = [result[0] for result in results]
moisturizers["policy_decision"] = [result[1] for result in results]
pd.crosstab(moisturizers["policy_rule"], moisturizers["policy_decision"], margins=True)

policy_decision,Exclude,Include,To check,All
policy_rule,,,,
Disagreement,0,0,31,31
Lina and Loredana agree,45,138,0,183
Moved from sunscreens,0,0,2,2
Needs more evidence,0,0,30,30
All,45,138,63,246


In [565]:
# the moisturizers to decide by hand: Lina's view and mine side by side
moist_to_check = moisturizers[moisturizers["policy_decision"] == "To check"].copy()
moist_to_check["link"] = "https://world.openbeautyfacts.org/product/" + moist_to_check["code"]
moist_to_check = moist_to_check.sort_values(["policy_rule", "france"], ascending=[True, False])
moist_to_check[["code", "product_name", "brands", "france", "eu", "policy_rule", "lina_decision", "lina_category",
                "loredana_decision", "loredana_reason", "conflict", "suggestion", "link"]].to_csv(
    RAW + "moisturizers_to_decide_v1.csv", index=False)
print(len(moist_to_check), "moisturizers to decide")
print(moist_to_check["policy_rule"].value_counts())

63 moisturizers to decide
policy_rule
Disagreement             31
Needs more evidence      30
Moved from sunscreens     2
Name: count, dtype: int64


In [566]:
# my decisions for the moisturizers checked by hand
MY_CHECK = {
    "5201143734776": ["Moisturizer, not France", "Diadermine Hydralist Aquagel, Greek label; list now split on the · separator"],
    "8809686383559": ["Keep", "nourishing Korean facial moisturizer, no parfum"],
    "3178040695153": ["Exclude", "INCI not split (2 items): signals wrong"],
    "3178040694149": ["Verify", "night cream but INCI not split (2 items): re-enter INCI"],
    "26023458": ["Keep", "face moisturizer with UV filters (octocrylene, avobenzone)"],
    "3145891410709": ["Exclude", "INCI broken (2 items)"],
    "3517360020793": ["Verify", "moisturizer for blemish-prone skin, INCI mixes French text: clean the list"],
    "3515450016527": ["Verify", "aroma = fragrance, product type unclear"],
    "00027606": ["Keep", "soothing face cream"],
    "8717763021662": ["Keep", "complete INCI (25), no parfum, face night cream"],
    "5000000000001": ["Keep", "Kazidomi dry and sensitive skin face cream, placeholder code, provisional"],
    "5000000000002": ["Keep", "Kazidomi combination to oily skin face cream, placeholder code, provisional"],
    "3473311503008": ["Keep", "daily anti-wrinkle face care, contains lavender oil"],
    "2021092899998": ["Keep", "day cream with UV filters, shop internal code"],
    "3660005148038": ["Exclude", "INCI incomplete (1 item)"],
    "3350031992612": ["Keep", "face and body cream, sold in France"],
    "3511720118915": ["Keep", "night face cream, perfumed"],
    "3511720117970": ["Keep", "day face cream, perfumed"],
    "3661467002128": ["Keep", "face cream"],
    "5612060068639": ["Keep", "anti-wrinkle face cream with UV filters"],
    "3401360003750": ["Keep", "day cream for dry skin"],
    "3253581145623": ["Keep", "face and body moisturizing cream, sold in France, 8 ml sample"],
    "3256224359418": ["Keep", "nourishing face cream"],
    "3361370627073": ["Keep", "cold cream, paraffin first, no parfum: very occlusive"],
    "3760171121023": ["Keep", "anti-age face cream"],
    "8684149131485": ["Exclude", "list too short"],
    "2000000157868": ["Exclude", "6 ingredients, product type unclear"],
    "8000137016884": ["Exclude", "ingredients written as an Italian description, not INCI"],
    "8710198421410": ["Exclude", "udder cream = hands and body cream"],
    "3600523201501": ["Exclude", "Revitalift filler = serum, INCI not split"],
    "4260189522816": ["Exclude", "no brand, product type unclear"],
    "3600551156699": ["Keep", "anti-dark spot care SPF 20: care first, so a moisturizer"],
    "4005900546272": ["Keep", "anti-wrinkle day cream SPF 30: care first, so a moisturizer"],
    "3282771001432": ["Verify", "INCI incomplete (no UV filters listed)"],
    "20434182": ["Keep", "anti-age day cream"],
    "3600550848359": ["Exclude", "face and body cream of a baby range"],
    "4550668265885": ["Verify", "Japanese barcode, contains retinol"],
    "3564700701058": ["Exclude", "face and body cream"],
    "3560070881673": ["Exclude", "face and body cream"],
    "5400606008694": ["Verify", "face and body cream, looks like a baby or mother range: check label"],
    "3511720118601": ["Exclude", "checked by hand"],
    "3511720118908": ["Keep", "day face cream, perfumed"],
    "3600541144019": ["Exclude", "serum-cream 2 in 1"],
    "8437018443420": ["Keep", "facial cream"],
    "8683561170119": ["Moisturizer, not France", "Turkish anti-spot cream (glycolic + azelaic acid)"],
    "8683561170126": ["Moisturizer, not France", "Turkish brightening cream"],
    "3600523473977": ["Keep", "day cream, Turkish product, retinoid-like retino peptide"],
    "8680540171259": ["Keep", "anti-spot cream SPF 15"],
    "8480017242679": ["Keep", "Spanish night cream, INCI starts with label text"],
    "8053732093729": ["Keep", "day and night face cream, ingredients field is a marketing text in Italian"],
    "8053630590009": ["Keep", "night cream with retinol"],
    "8001120595096": ["Moisturizer, not France", "Italian face cream"],
    "5021320154353": ["Exclude", "checked by hand"],
    "00012282": ["Moisturizer, not France", "Latin American anti-wrinkle cream"],
    "5391510478744": ["Keep", "face cream"],
    "42420125": ["Moisturizer, not France", "Nivea Men gel"],
    "8717953135704": ["Exclude", "for children"],
    "6260494210635": ["Moisturizer, not France", "Iranian cream"],
    "20237882": ["Keep", "certified natural facial moisturizer (German supermarket Norma)"],
    "8717333596309": ["Keep", "Dutch sebum-reducing cream with salicylic acid"],
    "4058172066153": ["Exclude", "baby cream"],
    "4010355141873": ["Exclude", "baby cream"],
    "4010355322852": ["Exclude", "German men day cream with UV filters"],
}

# my words -> the decision of the catalogue. All countries now: "not France" is still a moisturizer
CONVERT = {"Keep": "Include", "Moisturizer, not France": "Include", "Exclude": "Exclude", "Verify": "Verify"}

MOIST_MY_DECISIONS = {}
for code, (my_word, reason) in MY_CHECK.items():
    MOIST_MY_DECISIONS[code] = [CONVERT[my_word], "hand check 11/11", reason]

# every moisturizer to check has a decision, and every code is a moisturizer to check
assert len(MOIST_MY_DECISIONS) == len(moist_to_check), "a moisturizer to check has no decision"

moisturizers["my_decision"] = ""
moisturizers["my_rule"] = ""
moisturizers["my_reason"] = moisturizers["loredana_reason"].fillna("")
for code, (decision, rule, reason) in MOIST_MY_DECISIONS.items():
    assert code in list(moist_to_check["code"]), code + " is not a moisturizer to check"
    moisturizers.loc[moisturizers["code"] == code, ["my_decision", "my_rule", "my_reason"]] = [decision, rule, reason]


def moisturizer_final(row):
    """The agreed decision, my hand decision, or To Do while I haven't decided."""
    if row["policy_decision"] != "To check":
        return row["policy_decision"]
    if row["my_decision"] != "":
        return row["my_decision"]
    return "To Do"


moisturizers["final_decision"] = moisturizers.apply(moisturizer_final, axis=1)
moisturizers["inci_to_complete"] = False
print("Still to decide:", (moisturizers["final_decision"] == "To Do").sum())
pd.crosstab(moisturizers["final_decision"], moisturizers["france"], margins=True)

Still to decide: 0


france,False,True,All
final_decision,,,
Exclude,31,33,64
Include,100,76,176
Verify,0,6,6
All,131,115,246


In [567]:
# checks
assert len(moisturizers) == 244 + len(sun_moved), "a product was lost or added"
assert moisturizers["code"].is_unique, "a code appears twice"
wrong = moisturizers[~moisturizers["final_decision"].isin(["Include", "Exclude", "Verify", "To Do"])]
assert len(wrong) == 0, wrong["product_name"].tolist()

moisturizers.to_csv(RAW + "moisturizers_reviewed_v1.csv", index=False)
print("Saved:", len(moisturizers), "moisturizers")

Saved: 246 moisturizers


# Clean duplicates and generate the catalogue V2 output 

In [568]:
# Clean duplicates and generate the catalogue V2 output
cleansers_reviewed = pd.read_csv(RAW + "cleansers_reviewed_v1.csv", dtype={"code": str})
sunscreens_reviewed = pd.read_csv(RAW + "sunscreens_reviewed_v1.csv", dtype={"code": str})
moisturizers_reviewed = pd.read_csv(RAW + "moisturizers_reviewed_v1.csv", dtype={"code": str})
moisturizers_reviewed = moisturizers_reviewed.fillna({"my_decision": "", "my_rule": "", "my_reason": ""})

# sunscreens moved to the moisturizers are already in moisturizers_reviewed (part 4)
moved = sunscreens_reviewed["final_decision"] == "Move to moisturizer"
sunscreens_reviewed = sunscreens_reviewed[~moved]

cleansers_reviewed["inci_to_complete"] = False
# Balea hautrein: the ingredient list has no commas, so it can't be split
cleansers_reviewed.loc[cleansers_reviewed["product_name"] == "hautrein", "inci_to_complete"] = True

COLUMNS = ["role", "code", "product_name", "brands", "france", "eu", "ingredients_text",
           "policy_rule", "policy_decision", "my_rule", "my_decision", "final_decision", "my_reason",
           "inci_to_complete"]
catalogue = pd.concat([cleansers_reviewed[COLUMNS], moisturizers_reviewed[COLUMNS], sunscreens_reviewed[COLUMNS]],
                      ignore_index=True)

assert len(catalogue) == 494, "products lost or added"
assert catalogue["code"].is_unique, "a code appears twice"
pd.crosstab(catalogue["role"], catalogue["final_decision"], margins=True)

final_decision,Exclude,Include,Verify,All
role,,,,
cleanser,73,115,1,189
moisturizer,64,176,6,246
sunscreen,9,50,0,59
All,146,341,7,494


# split the ingredient lists

In [569]:
MIN_PRODUCTS_PER_INGREDIENT = 5

cosing = cosing.copy()
cosing["inci"] = cosing["inci_name"].str.lower()
cosing = cosing.drop_duplicates("inci")      # 3 names appear twice in the reference
inci_names = list(cosing["inci"])
print(len(inci_names), "INCI names in the reference")

1518 INCI names in the reference


In [570]:
def split_ingredients(text):
    """Turn the ingredient text into a list of names."""
    text = str(text).lower()
    text = text.replace("ingredients:", "").replace("ingrédients:", "")
    text = text.replace("•", ",").replace("·", ",")              # some labels use • or · instead of ,
    text = text.replace("1,2-hexanediol", "1-2-hexanediol")     # this comma is part of the name
    if "," not in text:
        text = text.replace(". ", ",").replace(" - ", ",")      # some labels use '. ' or ' - ' instead of ','
    names = []
    for piece in text.split(","):
        name = piece.strip().strip(".").strip()
        if name != "":
            names.append(name.replace("1-2-hexanediol", "1,2-hexanediol"))
    return names


def to_inci(name):
    """Clean one ingredient name and return its INCI name when we know it."""
    name = name.lower().replace("*", "").strip().strip(".").strip()
    if "(" in name and ")" in name:                      # 'parkii (shea) butter' -> 'parkii butter'
        name = name.split("(")[0] + name.split(")")[-1]
    elif "(" in name:                                    # 'aqua (water' (bracket not closed) -> 'aqua'
        name = name.split("(")[0]
    name = name.replace(")", "")                         # NEW 'ci 77499)' (bracket not opened) -> 'ci 77499'
    if name.endswith("%"):                               # NEW 'octocrylene 3%' -> 'octocrylene'
        name = " ".join(name.split()[:-1])               # NEW
    name = " ".join(name.split())
    name = ALIASES.get(name, name)
    if name not in inci_names and "/" in name:           # 'aqua/water/eau' -> 'aqua'
        first = ALIASES.get(name.split("/")[0].strip(), name.split("/")[0].strip())
        if first in inci_names:
            name = first
    return name

print(split_ingredients("Water · Glycerin · Aroma"))
print(to_inci("Aqua / Water"), "|", to_inci("butyrospermum parkii (shea) butter*"), "|", to_inci("aroma"))

['water', 'glycerin', 'aroma']
aqua | butyrospermum parkii butter | parfum


In [571]:
# one row per product and ingredient
rows = []
for code, text in zip(catalogue["code"], catalogue["ingredients_text"]):
    position = 1
    for raw_name in split_ingredients(text):
        rows.append({"code": code, "position": position, "raw_name": raw_name, "inci": to_inci(raw_name)})
        position = position + 1

long = pd.DataFrame(rows)
long = long[long["inci"] != ""]
long["in_cosing"] = long["inci"].isin(inci_names)

print("Rows:", len(long))
print("Different names before / after:", long["raw_name"].nunique(), "/", long["inci"].nunique())
print("Found in CosIng:", round(long["in_cosing"].mean() * 100, 1), "%")

Rows: 11076
Different names before / after: 2551 / 2180
Found in CosIng: 91.2 %


In [572]:
catalogue["inci_text"] = catalogue["code"].map(long.groupby("code")["inci"].apply(" | ".join)).fillna("")
catalogue["nb_ingredients"] = catalogue["code"].map(long.groupby("code")["inci"].count()).fillna(0).astype(int)
catalogue["coverage"] = catalogue["code"].map(long.groupby("code")["in_cosing"].mean()).fillna(0).round(2)

print("Lists shorter than", MIN_INGREDIENTS, "ingredients:")
print(catalogue[catalogue["nb_ingredients"] < MIN_INGREDIENTS]["role"].value_counts())
catalogue.groupby("role")["coverage"].mean().round(2)

Lists shorter than 5 ingredients:
role
moisturizer    13
cleanser       12
sunscreen       5
Name: count, dtype: int64


role
cleanser       0.87
moisturizer    0.87
sunscreen      0.80
Name: coverage, dtype: float64

# Signals based on Lina's and my MVP first outputs

In [573]:
# signals of the moisturizer notebook (names used by Lina and Loubna)
catalogue["has_parfum"] = catalogue["inci_text"].str.contains("parfum")
catalogue["has_fragrance_allergen"] = catalogue["inci_text"].str.contains("|".join(FRAGRANCE_ALLERGENS))
catalogue["has_essential_oil"] = catalogue["inci_text"].str.contains("|".join(ESSENTIAL_OILS))
catalogue["has_acne_active"] = catalogue["inci_text"].str.contains("|".join(ACNE_ACTIVES))

names = catalogue["product_name"].str.lower().fillna("")
catalogue["is_anti_ageing"] = names.str.contains("anti-âge|anti-age|rides|wrinkle|firming|lift")
catalogue["is_baby"] = names.str.contains("bébé|bebe|baby")
catalogue["is_face_and_body"] = names.str.contains("corps|body")


def count_heavy(ingredients):
    """How many ingredients of the list are heavy and can identify a rich texture?"""
    count = 0
    for ingredient in ingredients:
        for heavy in HEAVY_INGREDIENTS:
            if heavy in ingredient:
                count = count + 1
                break
    return count


def find_texture(inci_text, name):
    """Identify the texture rich, light or medium from the product name and the first ingredients"""
    name = str(name).lower()
    for word in RICH_NAME_WORDS:
        if word in name:
            return "rich"
    for word in LIGHT_NAME_WORDS:
        if word in name and "baume" not in name and "gelée" not in name:
            return "light"
    ingredients = inci_text.split(" | ")
    heavy_top3 = count_heavy(ingredients[:3])
    heavy_top5 = count_heavy(ingredients[:5])
    if ingredients[0] != "aqua" or heavy_top3 >= 1 or heavy_top5 >= 2:
        return "rich"
    elif heavy_top5 == 0:
        return "light"
    else:
        return "medium"


textures = [find_texture(text, name) for text, name in zip(catalogue["inci_text"], catalogue["product_name"])]
catalogue["is_rich"] = [texture == "rich" for texture in textures]
catalogue["is_light"] = [texture == "light" for texture in textures]

In [574]:
# new signals for the 3 concerns (routine proposal 10/11)
DRY_SKIN_ACTIVES = ["ceramide", "urea", "butyrospermum parkii butter", "squalane", "panthenol"]   # glycerin apart: in most products
RETINOIDS = ["retinol", "retinal", "retinyl", "retinoate", "retinaldehyde"]
STRONG_ACIDS = ["glycolic acid", "lactic acid", "mandelic acid"]
BANNED_EU = ["butylphenyl methylpropional"]          # Lilial, banned in the EU since March 2022
# UV_FILTERS: list of part 3


def top5(inci_text):
    """The first 5 ingredients (the biggest amounts)."""
    return " | ".join(inci_text.split(" | ")[:5])


catalogue["has_dry_skin_active"] = catalogue["inci_text"].str.contains("|".join(DRY_SKIN_ACTIVES))
catalogue["has_retinoid"] = catalogue["inci_text"].str.contains("|".join(RETINOIDS))
catalogue["has_strong_acid"] = catalogue["inci_text"].str.contains("|".join(STRONG_ACIDS))
catalogue["has_alcohol_denat"] = catalogue["inci_text"].str.contains("alcohol denat")
catalogue["has_uv_filter"] = catalogue["inci_text"].str.contains("|".join(UV_FILTERS))
catalogue["has_banned_eu"] = catalogue["inci_text"].str.contains("|".join(BANNED_EU))

# gentle = for sensitive / irritated skin
catalogue["is_gentle"] = ((catalogue["has_parfum"] == False) & (catalogue["has_essential_oil"] == False)
                          & (catalogue["has_alcohol_denat"] == False) & (catalogue["has_retinoid"] == False)
                          & (catalogue["has_strong_acid"] == False))

In [575]:
# vegan: "no animal-derived ingredient in the list" (not a vegan label)
ANIMAL_INGREDIENTS = ["cera alba", "beeswax", "propolis", "royal jelly", "lanolin", "collagen", "elastin",
                      "keratin", "silk", "sericin", "snail", "lactis", "casein", "goat milk", "donkey milk",
                      "carmine", "ci 75470", "guanine", "ci 75170", "shellac", "tallow", "chitosan",
                      "caviar", "mink oil"]
NOT_ANIMAL = ["vegan", "vegetable", "plant", "phyto"]
UNCERTAIN = ["squalane", "squalene", "cholesterol", "stearic acid"]


def has_animal(inci_text):
    ingredients = inci_text.split(" | ")
    for ingredient in ingredients:
        plant_version = False
        for word in NOT_ANIMAL:
            if word in ingredient:
                plant_version = True
        if plant_version:
            continue
        if ingredient == "mel" or "honey" in ingredient or "mel extract" in ingredient:
            return True
        for animal in ANIMAL_INGREDIENTS:
            if animal in ingredient:
                return True
    return False


def is_uncertain(inci_text):
    ingredients = inci_text.split(" | ")
    for ingredient in ingredients:
        for word in UNCERTAIN:
            if word in ingredient:
                return True
    return False


catalogue["has_animal_ingredient"] = catalogue["inci_text"].apply(has_animal)
catalogue["animal_uncertain"] = catalogue["inci_text"].apply(is_uncertain)
catalogue["is_vegan"] = (catalogue["has_animal_ingredient"] == False) & (catalogue["animal_uncertain"] == False)

In [576]:
# ingredient columns asked for the scoring (10/11): one INCI name each, so Loubna & Imane can weigh them herself
catalogue["has_salicylic_acid"] = catalogue["inci_text"].str.contains("salicylic acid")
catalogue["has_betaine_salicylate"] = catalogue["inci_text"].str.contains("betaine salicylate")
catalogue["has_glycerin"] = catalogue["inci_text"].str.contains("glycerin")
catalogue["has_petrolatum"] = catalogue["inci_text"].str.contains("petrolatum")
catalogue["has_panthenol"] = catalogue["inci_text"].str.contains("panthenol")
catalogue["has_ceramides"] = catalogue["inci_text"].str.contains("ceramide")

# sensitive claim = the name says "sensitive skin" (a claim, not a proof of tolerance)
catalogue["sensitive_claim"] = names.str.contains("sensible|sensitive|sensitiv|hassas")

# rinse mode: cleansers are rinsed off, except micellar waters, cleansing milks and "no rinse" products
NO_RINSE_WORDS = ["micel", "h2o", "h20", "eau micell", "eau démaquill", "cleansing water", "lait", "milk",
                  "sans rinçage", "no rinse", "no-rinse"]


def rinse_mode(row):
    if row["role"] != "cleanser":
        return "leave-on"                    # moisturizers and sunscreens stay on the skin
    name = str(row["product_name"]).lower()
    if has_word(name, NO_RINSE_WORDS):
        return "leave-on"
    return "rinse-off"


catalogue["rinse_mode"] = catalogue.apply(rinse_mode, axis=1)

# ingredient list status: incomplete = list too short or to complete from the label
catalogue["inci_status"] = "complete"
catalogue.loc[(catalogue["nb_ingredients"] < MIN_INGREDIENTS) | catalogue["inci_to_complete"], "inci_status"] = "incomplete"

print(pd.crosstab(catalogue["role"], catalogue["rinse_mode"]))
print(catalogue["inci_status"].value_counts())

rinse_mode   leave-on  rinse-off
role                            
cleanser           51        138
moisturizer       246          0
sunscreen          59          0
inci_status
complete      462
incomplete     32
Name: count, dtype: int64


In [577]:
# pregnancy (proposal 10/11): no referral any more, products WITHOUT these ingredients + a message
# "this is only a suggestion, ask your doctor or pharmacist". Sources: AAD, Nebraska Medicine (retinoids,
# hydroquinone). Essential oils: Loredana's decision based on knowledge.
PREGNANCY_AVOID = ["retinol", "retinal", "retinyl", "retinoate", "retinaldehyde",   # cosmetic retinoids
                   "tretinoin", "adapalene", "tazarotene", "isotretinoin",          # prescription retinoids
                   "hydroquinone"]
catalogue["has_pregnancy_avoid"] = catalogue["inci_text"].str.contains("|".join(PREGNANCY_AVOID))
catalogue["has_oxybenzone"] = catalogue["inci_text"].str.contains("benzophenone-3|oxybenzone")
catalogue["is_pregnancy_ok"] = (catalogue["has_pregnancy_avoid"] == False) & (catalogue["has_essential_oil"] == False)
print("OK for pregnancy:", catalogue["is_pregnancy_ok"].sum(), "/", len(catalogue))

OK for pregnancy: 416 / 494


In [578]:
SIGNALS = ["has_parfum", "has_fragrance_allergen", "has_essential_oil", "has_acne_active", "is_rich", "is_light",
           "is_anti_ageing", "is_baby", "is_face_and_body", "has_dry_skin_active", "has_retinoid",
           "has_strong_acid", "has_alcohol_denat", "has_uv_filter", "has_banned_eu", "is_gentle",
           "has_animal_ingredient", "animal_uncertain", "is_vegan", "has_salicylic_acid",
           "has_betaine_salicylate", "has_glycerin", "has_petrolatum", "has_panthenol", "has_ceramides",
           "sensitive_claim", "has_pregnancy_avoid", "has_oxybenzone", "is_pregnancy_ok"]

# % of products with each signal, per step
(catalogue.groupby("role")[SIGNALS].mean() * 100).round(0).T

role,cleanser,moisturizer,sunscreen
has_parfum,47.0,67.0,42.0
has_fragrance_allergen,24.0,44.0,29.0
has_essential_oil,13.0,13.0,8.0
has_acne_active,20.0,22.0,20.0
is_rich,35.0,40.0,27.0
is_light,63.0,50.0,71.0
is_anti_ageing,2.0,15.0,7.0
is_baby,0.0,1.0,0.0
is_face_and_body,0.0,3.0,10.0
has_dry_skin_active,35.0,56.0,31.0


# Relunch the duplicates based on Lina's notebook and keep the columns

In [579]:
catalogue["name_brand"] = (catalogue["product_name"].str.lower().fillna("") + " | "
                           + catalogue["brands"].str.lower().fillna(""))
catalogue["possible_duplicate"] = catalogue.duplicated(["role", "name_brand"], keep=False)
print("Same name + brand:", catalogue["possible_duplicate"].sum(), "products")

# 2. same ingredient list in the same step, but another name = to check by hand (same formula, or a data error?)
long_enough = catalogue["nb_ingredients"] >= MIN_INGREDIENTS
same_inci = catalogue.duplicated(["role", "inci_text"], keep=False) & long_enough & ~catalogue["possible_duplicate"]
catalogue["same_inci_other_name"] = same_inci
for index, row in catalogue[same_inci].sort_values("inci_text").iterrows():
    print(row["role"], "|", row["code"], "|", row["france"], "|", row["final_decision"], "|", row["product_name"], "|", row["brands"])

Same name + brand: 16 products
moisturizer | 4262351040319 | False | Include | Rhodophyta Spirulina Hydrasafe Night Cream | caremare
moisturizer | 4262351040296 | False | Include | RHODOPHYTA ALOE VERA SHIELD DAY CREAM | caremare
moisturizer | 8699956514185 | False | Include | Bioderma Atoderm Cream Ultra | Bioderma
moisturizer | 8699956513676 | False | Include | Bioderma Atoderm Creme Ultra | Bioderma
moisturizer | 3560071162115 | True | Include | crème de soin hydratante | Carrefour
moisturizer | 3560070881673 | True | Exclude | Crème Visage et corps | Carrefour
cleanser | 7640183490286 | False | Exclude | Allergic Skin Cleansing Bar - Bag | Swiss Toniq Geneva
cleanser | 7640183490262 | False | Include | Make-Up Remover Cleansing Bar - Solid Box | Swiss Toniq Geneva
cleanser | 7640183490699 | False | Exclude | Allergic Skin Cleansing Bar - Solid Box | Swiss Toniq Geneva
cleanser | 7640183490675 | False | Exclude | Make-Up Remover Cleansing Bar - Bag | Swiss Toniq Geneva
cleanser | 76

In [580]:
# keep ONE product per name + brand: France first, then the longest ingredient list, then the best CosIng coverage
ordered = catalogue.sort_values(["france", "nb_ingredients", "coverage"], ascending=False)
kept_code = ordered.groupby(["role", "name_brand"])["code"].first()        # the code we keep in each group

def duplicate_of(row):
    """The code of the product we keep, or '' when this product is the one we keep."""
    kept = kept_code[(row["role"], row["name_brand"])]
    if kept == row["code"]:
        return ""
    return kept

catalogue["duplicate_of"] = catalogue.apply(duplicate_of, axis=1)
catalogue["is_duplicate_v2"] = catalogue["duplicate_of"] != ""
print("Duplicates removed from the recommendations:", catalogue["is_duplicate_v2"].sum())
catalogue[catalogue["possible_duplicate"]].sort_values(["role", "name_brand"])[
    ["role", "code", "france", "nb_ingredients", "product_name", "duplicate_of"]]

Duplicates removed from the recommendations: 8


,role,code,france,nb_ingredients,product_name,duplicate_of
169,cleanser,3574661214771,True,17,Deep Clean,3574661215341
170,cleanser,3574661215341,True,18,Deep Clean,
364,moisturizer,4058172066153,False,22,leichte Gesichtscreme,
365,moisturizer,4010355141873,False,22,leichte Gesichtscreme,4058172066153
221,moisturizer,7501054500193,False,15,Nivea creme,4005800001192
273,moisturizer,4005800001192,True,21,Nivea Creme,
189,moisturizer,0860015354624,False,6,PLU Laboratories GHK-Cu Face Tonic,
190,moisturizer,0860015354648,False,6,PLU Laboratories GHK-Cu Face Tonic,0860015354624
297,moisturizer,3600523201501,False,7,Revitalift filler,3600523732548
375,moisturizer,3600523732548,True,13,Revitalift filler,


In [581]:
# in_scope = the step decision says Include (no country filter: France / EU are filtered by the app)
catalogue["in_scope"] = catalogue["final_decision"] == "Include"

# recommendable = in scope + a usable ingredient list + no ingredient banned in the EU + not a duplicate
catalogue["is_recommendable"] = (catalogue["in_scope"]
                                 & (catalogue["nb_ingredients"] >= MIN_INGREDIENTS)
                                 & (catalogue["inci_to_complete"] == False)
                                 & (catalogue["has_banned_eu"] == False)
                                 & (catalogue["is_duplicate_v2"] == False))

pd.crosstab(catalogue["role"], [catalogue["france"], catalogue["is_recommendable"]], margins=True)

france           False       True       All
is_recommendable False True False True     
role                                       
cleanser            38   62    42   47  189
moisturizer         39   92    41   74  246
sunscreen           10   36     6    7   59
All                 87  190    89  128  494

In [582]:
# Lina's catalogue columns to harmonize the output
lina_columns = []
for column in lina_keep.columns:
    if column not in catalogue.columns:
        lina_columns.append(column)
catalogue = catalogue.merge(lina_keep[["code"] + lina_columns], on="code", how="left")
print(len(lina_columns), "columns of Lina added")

# ingredient matrix: one column per ingredient found in at least 5 products, 1 = the product contains it
count = long.groupby("inci")["code"].nunique()
frequent = list(count[count >= MIN_PRODUCTS_PER_INGREDIENT].index)
frequent_rows = long[long["inci"].isin(frequent)]
matrix = pd.crosstab(frequent_rows["code"], frequent_rows["inci"])
matrix = (matrix > 0).astype(int)

# keep every column of Lina's matrix, even the rare ingredients
for column in lina_matrix.columns:
    if column != "code" and column not in matrix.columns:
        matrix[column] = long[long["inci"] == column].groupby("code").size().reindex(matrix.index).fillna(0).clip(upper=1).astype(int)
matrix = matrix.reindex(catalogue["code"]).fillna(0).astype(int)      # one row per product, same order
matrix.index.name = "code"
print("Matrix:", matrix.shape)

31 columns of Lina added
Matrix: (494, 363)


In [583]:
# fill Lina's review columns for the products she did not review (cleansers, sunscreens, other moisturizers),
# with the same meaning. Her own values are never changed: fillna only fills the empty cells.
catalogue[lina_columns] = catalogue[lina_columns].astype(object)   # text or True/False in the same column
rule_names = {}
for index, row in policy.iterrows():
    rule_names[row["rule_id"]] = row["product_category"]
for index, row in sun_policy.iterrows():
    rule_names[row["rule_id"]] = row["product_category"]


def rule_id(row):
    """The rule that decided: my rule when checked by hand, otherwise the rule of the policy."""
    if row["policy_decision"] == "To check":
        return row["my_rule"]
    return row["policy_rule"]


catalogue["rule_id"] = catalogue.apply(rule_id, axis=1)
by_hand = catalogue["policy_decision"] == "To check"
decided = catalogue["final_decision"].isin(["Include", "Exclude"])
incomplete = catalogue["inci_status"] == "incomplete"

# Lina's words for the decisions
LINA_CLASSIFICATION = {"Include": "Keep", "Exclude": "Exclude", "Verify": "Needs more evidence", "To Do": "To decide"}
quality_flags = pd.Series("", index=catalogue.index)
quality_flags[incomplete] = "ingredient_list_completeness_check"
quality_flags[catalogue["has_banned_eu"]] = (quality_flags[catalogue["has_banned_eu"]] + "; banned_eu_ingredient").str.strip("; ")

exclusion = pd.Series("", index=catalogue.index)
exclusion[catalogue["final_decision"] == "Exclude"] = catalogue["rule_id"] + " " + catalogue["rule_id"].map(rule_names).fillna("")
exclusion[catalogue["is_duplicate_v2"]] = "duplicate"
exclusion[(catalogue["final_decision"] == "Include") & (catalogue["nb_ingredients"] < MIN_INGREDIENTS)] = "list too short"

FILL = {
    "exclusion_reason": exclusion.replace("", pd.NA),
    "is_duplicate": catalogue["is_duplicate_v2"],
    "to_check_by_hand": by_hand,
    "category_status": by_hand.map({True: "Needs manual review", False: "Decided by rule"}),
    "category_status_v1": by_hand.map({True: "Needs manual review", False: "Decided by rule"}),
    "product_category": catalogue["rule_id"].map(rule_names),
    "final_category": catalogue["rule_id"].map(rule_names),
    "name_normalized": catalogue["product_name"].str.lower().str.strip(),
    "brand_normalized": catalogue["brands"].str.lower().str.split(",").str[0].str.strip(),
    "review_decision": catalogue["my_decision"].replace("", pd.NA),
    "review_notes": catalogue["my_reason"].replace("", pd.NA),
    "review_flags_updated": quality_flags.replace("", pd.NA),
    "all_review_flags": quality_flags.replace("", pd.NA),
    "review_status": by_hand.map({True: "Reviewed by hand", False: "Decided by rule"}),
    "basic_quality_audit": incomplete.map({True: "Ingredient list incomplete", False: "Passed basic checks"}),
    "duplicate_partner_code": catalogue["duplicate_of"].replace("", pd.NA),
    "candidate_status": catalogue["final_decision"].map(LINA_CLASSIFICATION),
    "ingredient_matrix_status": "Matched",
    "final_review_classification": catalogue["final_decision"].map(LINA_CLASSIFICATION),
    "review_source_file": "loredana_catalogue_v2 (" + catalogue["role"] + " policy v1)",
    "review_policy_annotation": catalogue["rule_id"].replace("", pd.NA),
    "review_evidence_notes": catalogue["my_reason"].replace("", pd.NA),
    "review_quality_flags": quality_flags.replace("", pd.NA),
    "review_quality_followup_needed": incomplete | catalogue["has_banned_eu"] | (catalogue["final_decision"] == "Verify"),
    "moisturizer_inclusion_approved": (catalogue["role"] == "moisturizer") & (catalogue["final_decision"] == "Include"),
    "requires_classification_evidence": catalogue["final_decision"] == "Verify",
    "catalogue_review_complete": decided,
}
not_lina = ~catalogue["code"].isin(lina_keep["code"])          # Lina's 152 rows are not touched
for column in FILL:
    values = FILL[column]
    if type(values) == str:                                    # one text for every row
        values = pd.Series(values, index=catalogue.index)
    catalogue.loc[not_lina, column] = catalogue.loc[not_lina, column].fillna(values[not_lina])

# duplicates found in v2: same words as Lina
dup = catalogue["is_duplicate_v2"] & not_lina
catalogue.loc[dup, "deduplication_status"] = "Same name and brand - one record kept (France first)"
catalogue.loc[dup, "duplicate_resolution_action"] = "Keep the record in duplicate_partner_code"
catalogue.loc[dup, "duplicate_verification_required"] = False
catalogue.loc[dup, "duplicate_merged"] = False

# new facts on Lina's rows, shown (not written) so she can decide in part 4
lina_rows = catalogue[~not_lina]
print("Lina's moisturizers with an EU-banned ingredient:", lina_rows[lina_rows["has_banned_eu"]]["product_name"].tolist())
print("Lina's moisturizers that are a v2 duplicate:", lina_rows[lina_rows["is_duplicate_v2"]]["product_name"].tolist())

empty = catalogue[lina_columns].isna().sum()
print("Empty cells left in Lina's columns:")
print(empty[empty > 0])

Lina's moisturizers with an EU-banned ingredient: ["L'Oréal Paris Yaş Uzmanı 30+ Kırışıklık Karşıtı Nemlendirici Krem", 'Bio-Hydratante', 'Idraet thermal new', 'Cell active hidro cream', 'DAILY ESSENTIALS LIGHT MOISTURISING DAY CREAM', 'Gel Revitalisant']
Lina's moisturizers that are a v2 duplicate: ['Nivea creme', 'Vitamin B, C and E moisturizer', 'Revitalift filler', 'Ultra facial cream']
Empty cells left in Lina's columns:
exclusion_reason                   341
product_category                    95
review_flags_updated               383
brand_normalized                     4
final_category                     180
review_decision                    336
review_notes                       311
all_review_flags                   383
deduplication_status               482
duplicate_partner_code             482
duplicate_resolution_action        482
duplicate_verification_required    482
duplicate_merged                   482
review_policy_annotation            42
review_evidence_notes   

/var/folders/bl/bvf3l2w16sb0jmy2v926fp0h0000gn/T/ipykernel_65322/827128064.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  catalogue.loc[not_lina, column] = catalogue.loc[not_lina, column].fillna(values[not_lina])
/var/folders/bl/bvf3l2w16sb0jmy2v926fp0h0000gn/T/ipykernel_65322/827128064.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  catalogue.loc[not_lina, column] = catalogue.loc[not_lina, column].fillna(values[not_lina])
/var/folders/bl/bvf3l2w16sb0jmy2v926fp0h0000gn/T/ipykernel_65322/827128064.py:68: FutureWarning: Downcasting object dtype 

In [584]:
# align the status columns on the LATEST status of each product (all rows, Lina's included).
# Lina's history stays in her own file moisturizer_catalogue_keep_reviewed_v1.csv (not changed).

def latest_decision(row):
    """Include / Exclude / Verify when decided, else To Do."""
    if row["final_decision"] in ["Include", "Exclude", "Verify"]:
        return row["final_decision"]
    return "To Do"


def latest_review(row):
    """Who decided last."""
    if row["final_decision"] in ["Include", "Exclude", "Verify"]:
        if row["policy_decision"] == "To check":
            return "Checked by hand (Loredana)"
        if row["role"] == "moisturizer":
            return "Lina and Loredana agree"
        return "Decided by rule (Loredana)"
    return "To decide (part 4)"


STATUS_WORDS = {"Include": "Confirmed", "Exclude": "Excluded", "Verify": "Needs more evidence", "To Do": "To decide"}

catalogue["latest_decision"] = catalogue.apply(latest_decision, axis=1)
catalogue["review_status"] = catalogue.apply(latest_review, axis=1)
catalogue["category_status"] = catalogue["latest_decision"].map(STATUS_WORDS)
catalogue["candidate_status"] = catalogue["latest_decision"].map(STATUS_WORDS)
catalogue["review_decision"] = catalogue["latest_decision"]
catalogue["final_review_classification"] = catalogue["latest_decision"].map(LINA_CLASSIFICATION)
catalogue["basic_quality_audit"] = incomplete.map({True: "Ingredient list incomplete", False: "Passed basic checks"})
catalogue["to_check_by_hand"] = catalogue["latest_decision"].isin(["Verify", "To Do"])   # still open
catalogue["catalogue_review_complete"] = catalogue["latest_decision"].isin(["Include", "Exclude"])

# the rule's own answer: "To check" means the rule could not decide and the product was checked by hand
catalogue["policy_decision"] = catalogue["policy_decision"].replace("To check", "Checked by hand")

print(pd.crosstab(catalogue["role"], catalogue["category_status"], margins=True))
print(catalogue["review_status"].value_counts())

category_status  Confirmed  Excluded  Needs more evidence  All
role                                                          
cleanser               115        73                    1  189
moisturizer            176        64                    6  246
sunscreen               50         9                    0   59
All                    341       146                    7  494
review_status
Decided by rule (Loredana)    215
Lina and Loredana agree       183
Checked by hand (Loredana)     96
Name: count, dtype: int64


In [585]:
# checks
assert len(catalogue) == 494, "products lost or added"
assert catalogue["code"].is_unique, "a code appears twice"

missing = []
for column in lina_keep.columns:
    if column not in catalogue.columns:
        missing.append(column)
assert missing == [], "columns removed from the catalogue: " + str(missing)

missing = []
for column in lina_matrix.columns:
    if column != "code" and column not in matrix.columns:
        missing.append(column)
assert missing == [], "columns removed from the matrix: " + str(missing)

wrong = catalogue[(catalogue["is_vegan"] == True) & (catalogue["has_animal_ingredient"] == True)]
assert len(wrong) == 0, wrong["product_name"].tolist()

recommendable = catalogue[catalogue["is_recommendable"]]
assert recommendable["name_brand"].duplicated().sum() == 0, "the same product is recommendable twice"
assert (recommendable["nb_ingredients"] >= MIN_INGREDIENTS).all()
sun_ok = recommendable[recommendable["role"] == "sunscreen"]
assert sun_ok["has_uv_filter"].all(), sun_ok[~sun_ok["has_uv_filter"]]["product_name"].tolist()

catalogue.to_csv(RAW + "catalogue_v2.csv", index=False)
matrix.to_csv(RAW + "ingredient_matrix_v2.csv")
print("Saved: catalogue_v2.csv", catalogue.shape, "| ingredient_matrix_v2.csv", matrix.shape)

Saved: catalogue_v2.csv (494, 88) | ingredient_matrix_v2.csv (494, 363)


In [586]:
import os
print("Part 4 file exists:", os.path.exists(RAW + "moisturizers_reviewed_v1.csv"))
print("Moisturizers in part 5a:", "moisturizers_reviewed" in globals())
print("Align cell ran:", "latest_decision" in catalogue.columns)
print(catalogue[catalogue["role"] == "moisturizer"]["final_decision"].value_counts())

Part 4 file exists: True
Moisturizers in part 5a: True
Align cell ran: True
final_decision
Include    176
Exclude     64
Verify       6
Name: count, dtype: int64
